### Regression problem to predict salary

In [2]:
#importing libraries

import numpy as np
import pandas as pd
import pickle, datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [3]:
#importing dataset into dataframe

data = pd.read_csv("churn_modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
#dropping columns RowNumber, CustomerId, Surname

data = data.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
#splitting data into train and test set.

x = data.drop(columns=['EstimatedSalary'])
y = data['EstimatedSalary']


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(7500, 10)
(2500, 10)
(7500,)
(2500,)


In [6]:
x_train['Gender'].head()

4901      Male
4375      Male
6698    Female
9805      Male
1101      Male
Name: Gender, dtype: str

In [7]:
#encoding categorical feature 'Gender'

label_encoder = LabelEncoder()

x_train['Gender'] = label_encoder.fit_transform(x_train['Gender'])
x_test['Gender'] = label_encoder.transform(x_test['Gender'])

In [8]:
x_train['Gender'].head()

4901    1
4375    1
6698    0
9805    1
1101    1
Name: Gender, dtype: int64

In [9]:
#encoding categorical feature 'Geography'

onehot_encoder = OneHotEncoder()

geography_train_arr = onehot_encoder.fit_transform(x_train[['Geography']]).toarray()
geography_test_arr = onehot_encoder.transform(x_test[['Geography']]).toarray()

geo_train_df = pd.DataFrame(geography_train_arr, columns=onehot_encoder.get_feature_names_out())
geo_test_df = pd.DataFrame(geography_test_arr, columns=onehot_encoder.get_feature_names_out())


In [10]:
train_df = pd.concat([x_train.reset_index(drop=True),geo_train_df], axis=1)
train_df = train_df.drop(columns=['Geography'])
train_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,673,1,59,0,178058.06,2,0,1,1,1.0,0.0,0.0
1,850,1,41,8,60880.68,1,1,0,0,0.0,1.0,0.0
2,725,0,31,6,0.00,1,0,0,0,1.0,0.0,0.0
3,644,1,33,7,174571.36,1,0,1,0,1.0,0.0,0.0
4,703,1,29,9,0.00,2,1,0,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
7495,768,1,54,8,69712.74,1,1,1,0,1.0,0.0,0.0
7496,682,0,58,1,0.00,1,1,1,0,1.0,0.0,0.0
7497,735,0,38,1,0.00,3,0,0,1,1.0,0.0,0.0
7498,667,1,43,8,190227.46,1,1,0,1,1.0,0.0,0.0


In [11]:
test_df = pd.concat([x_test.reset_index(drop=True),geo_test_df], axis=1)
test_df = test_df.drop(columns=['Geography'])
test_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,596,1,32,3,96709.07,2,0,0,0,0.0,1.0,0.0
1,623,1,43,1,0.00,2,1,1,0,1.0,0.0,0.0
2,601,0,44,4,0.00,2,1,0,0,0.0,0.0,1.0
3,506,1,59,8,119152.10,2,1,1,0,0.0,1.0,0.0
4,560,0,27,7,124995.98,1,1,1,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2495,645,0,55,1,133676.65,1,0,1,0,0.0,0.0,1.0
2496,569,0,51,3,0.00,3,1,0,1,0.0,0.0,1.0
2497,768,1,25,0,78396.08,1,1,1,0,1.0,0.0,0.0
2498,690,0,36,6,110480.48,1,0,0,0,1.0,0.0,0.0


In [12]:
#scaling the input features

scaler = StandardScaler()

x_train_arr = scaler.fit_transform(train_df)
x_test_arr = scaler.transform(test_df)

In [13]:
#saving ojects to pickle files

with open("onehot_encoder_regression.pkl", "wb") as file_obj:
    pickle.dump(onehot_encoder,file_obj)

with open("scaler_regression.pkl", "wb") as file_obj:
    pickle.dump(scaler, file_obj)

with open("label_encoder_regression.pkl", "wb") as file_obj:
    pickle.dump(label_encoder, file_obj)
        

In [14]:
#regression model building

model = Sequential([
    Dense(units=64, activation="relu", input_shape=(x_train_arr.shape[1],)),
    Dense(units=32, activation="relu"),
    Dense(units=1) #linear activation
])

model.summary()



d:\End_To_End_Projects\Customer_Churn_Classification\churnenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
#compiling model

model.compile(
    optimizer="adam",
    loss="mae",
    metrics=['accuracy']
)

In [16]:
#callbacks

tensorboard_callback = TensorBoard(log_dir="reglog/fit/"+datetime.datetime.now().strftime("%Y-%m-%d_%H_%M_%S"), histogram_freq=1)

earlystopping_callback = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, mode="min")


In [17]:
#model training

model.fit(x_train_arr, y_train, validation_data=(x_test_arr, y_test), epochs=100, verbose=True, callbacks=[tensorboard_callback, earlystopping_callback])

Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0000e+00 - loss: 100621.7656 - val_accuracy: 0.0000e+00 - val_loss: 98171.1719
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 99952.8359 - val_accuracy: 0.0000e+00 - val_loss: 96811.8750
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 97543.7031 - val_accuracy: 0.0000e+00 - val_loss: 93218.8203
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0000e+00 - loss: 92631.5234 - val_accuracy: 0.0000e+00 - val_loss: 87037.9453
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 85321.9688 - val_accuracy: 0.0000e+00 - val_loss: 78971.8516
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 76560.8672 - val_accuracy: 0.0000e+00 - val_loss: 70352.0312
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 67743.4062 - val_accuracy: 0.0000e+00 - val_l

In [ ]:
%load_ext tensorboard

%tensorboard --logdir "reglog/fit"

In [21]:
#evaluating model on test dataset

evaluation_loss, evalution_accuracy = model.evaluate(x_test_arr, y_test)
print(f"Evalution Loss: {evaluation_loss}")

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.0000e+00 - loss: 49974.9961  
Evalution Loss: 49974.99609375


In [22]:
#saving model in .keras file

model.save("reg_model.keras")